
# Estimation of DWPA model parameters by grid search and gradient-based optimisation

The code below estimates parameters of the DWPA model using grid search and a
quasi-Newton optimisation method, LBFGS.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joacorapela/lds_itba_2026/blob/main/code/notebooks/exercises/plotMLlbfgsTMP1.ipynb)

## Import packages



In [1]:
import time
import math
import numpy as np
import torch
from tqdm import tqdm
import plotly.graph_objs as go

In [2]:
torch.__version__

'2.15.0+cu132'

In [3]:
# pip install --upgrade torch

In [4]:
torch.__version__

'2.15.0+cu132'

## Function to compute the log-likelihood of LDS model parameters



In [5]:
def compute_log_like(y, A, Q, C, R, m0, V0):
    N = y.shape[1]
    P = C.shape[0]

    # Initial log-likelihood scalar/tensor
    log_like = torch.tensor(
        -0.5 * N * P * math.log(2 * math.pi),
        dtype=torch.double,
    )

    for n in range(N):
        if n == 0:
            # Out-of-place assignments for initial step
            xnn1 = A @ torch.unsqueeze(m0, -1)
            Pnn1 = A @ V0 @ A.T + Q
        else:
            xnn1 = A @ xnn
            Pnn1 = A @ Pnn @ A.T + Q

        Stmp = C @ Pnn1 @ C.T + R
        Sn = (Stmp + Stmp.T) / 2

        # Solve for inverse cleanly
        Sinv = torch.linalg.solve(
            Sn, torch.eye(P, dtype=torch.double),
        )

        K = Pnn1 @ C.T @ Sinv

        pred_obs = C @ xnn1

        # Handle observation slice dimensions safely
        yn = y[:, n:(n + 1)]  # keeps shape [P, 1]
        innov = yn - pred_obs

        xnn = xnn1 + K @ innov
        Pnn = Pnn1 - K @ C @ Pnn1

        # Out-of-place log-likelihood update
        log_like = log_like - 0.5 * (
            torch.logdet(Sn) + innov.T @ torch.linalg.solve(Sn, innov)
        )

    return log_like

## Set hyperparameters



In [6]:
pos_x0 = 0.0
vel_x0 = 0.0
ace_x0 = 0.0
dt = 1e-3
# num_pos = 1000
num_pos = 100
sigma_a = 1.0
sigma_y = 1.0
sqrt_diag_V0_value = 1e-03

## Set LDS parameters



In [7]:
A = torch.tensor([[1, dt, dt**2/2],
                  [0, 1, dt],
                  [0, 0, 1]],
                 dtype=torch.double)
C = torch.tensor([[1, 0, 0]], dtype=torch.double)

Qe = torch.tensor([[dt**4/4, dt**3/2, dt**2/2],
                   [dt**3/2, dt**2,   dt],
                   [dt**2/2, dt,      1]],
                  dtype=torch.double)
Q = sigma_a**2 * Qe
R = torch.diag(torch.tensor([sigma_y**2]))
m0 = torch.tensor([pos_x0, vel_x0, ace_x0],
                 dtype=torch.double)
V0 = torch.diag(torch.ones_like(m0)*sqrt_diag_V0_value**2)

## Sample from the LDS



In [8]:
start_time = time.time()

# fix random seed
np.random.seed(seed=1234)

M = A.shape[0]
P = C.shape[0]

# state noise
w = np.random.multivariate_normal(np.zeros(M), Q.numpy(), num_pos).T
# measurement noise
v = np.random.multivariate_normal(np.zeros(P), R.numpy(), num_pos).T
# initial state noise
x0 = np.random.multivariate_normal(m0.numpy(), V0.numpy(), 1).flatten()
# sample
x = np.empty(shape=(M, num_pos))
y = np.empty(shape=(P, num_pos))
x[:, 0] = A.numpy() @ x0 + w[:, 0]
for n in range(1, num_pos):
    x[:, n] = A.numpy() @ x[:, n-1] + w[:, n]
y = C.numpy() @ x + v
t = np.arange(num_pos) * dt

elapsed_time = time.time() - start_time

y = torch.from_numpy(y)
t = torch.from_numpy(t)

## Run a quasi-Newton (LBFGS) maximisation



In [9]:
# Define variables that control the optimisation

max_epochs = 30
max_iter_per_epoch = 10
lr = .05
tolerance_grad = 1e-7
tolerance_change = 1e-9
# line_search_fn = "strong_wolfe"
line_search_fn = None

# Set initial conditions
sigma_a0 = 39.0
sigma_y0 = 1.1

# perform optimisation
optim_params = {
    "lr": lr,
    "max_iter": max_iter_per_epoch,
    "tolerance_grad": tolerance_grad,
    "tolerance_change": tolerance_change,
    "line_search_fn": line_search_fn,
}
sigma_a_tensor = torch.tensor([sigma_a0], dtype=torch.double)
sigma_y_tensor = torch.tensor([sigma_y0], dtype=torch.double)
sigma_a_tensor.requires_grad = True
sigma_y_tensor.requires_grad = True
optimizer = torch.optim.LBFGS([sigma_a_tensor, sigma_y_tensor], **optim_params)


def closure():
    optimizer.zero_grad()

    R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
    Q_lbfgs = sigma_a_tensor**2 * Qe
    log_like = compute_log_like(
        y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
    )
    loss = -log_like
    loss.backward(retain_graph=True)
    return loss


# Run LBFGS optimisation
print(f"{'Iteration':<10} | {'Loss (Neg Log-Likelihood)':<25} | {'sigma_a':<12} | {'sigma_y':<12}")
print("-" * 65)

R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
Q_lbfgs = sigma_a_tensor**2 * Qe
ll0 = compute_log_like(
    y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
)

hist_ll = [ll0]
hist_sigma_a = [sigma_a0]
hist_sigma_y = [sigma_y0]
for epoch in range(max_epochs):
    # Perform optimization step
    optimizer.step(closure)

    # Save history of parameters and log-likelihoods
    R_lbfgs = (sigma_y_tensor**2).reshape(P, P)
    Q_lbfgs = sigma_a_tensor**2 * Qe
    lln = compute_log_like(
        y=y, A=A, Q=Q_lbfgs, C=C, R=R_lbfgs, m0=m0, V0=V0
    )

    hist_ll.append(lln)
    hist_sigma_a.append(sigma_a_tensor.item())
    hist_sigma_y.append(sigma_y_tensor.item())

    # Print progress metrics
    print(f"{epoch+1:<10} | {lln.item():<25.4f} |"
          f" {sigma_a_tensor.item():<12.4f} | {sigma_y_tensor.item():<12.4f}")

Iteration  | Loss (Neg Log-Likelihood) | sigma_a      | sigma_y     
-----------------------------------------------------------------
1          | -138.9128                 | 38.9994      | 0.9747      
2          | -138.8958                 | 38.9985      | 0.9657      
3          | -138.8938                 | 38.9974      | 0.9627      
4          | -138.8936                 | 38.9962      | 0.9616      
5          | -138.8935                 | 38.9944      | 0.9611      
6          | -138.8934                 | 38.9886      | 0.9604      
7          | -138.8930                 | 38.9571      | 0.9592      
8          | -138.8907                 | 38.7732      | 0.9562      
9          | -138.8770                 | 37.6699      | 0.9489      
10         | -138.7755                 | 29.5701      | 0.9263      
11         | -138.8034                 | 27.2869      | 0.9159      
12         | -138.7441                 | 23.1539      | 0.9089      
13         | -138.6728               